In [1]:
import sqlite3
import pandas as pd
import os
import csv

In [32]:
DATABASE = "../../../../v04_verb-case_pattern/drive_data/v33_koondkorpus_transaktsioonid_v04_2.db"

LINE_DATA_TABLE = "lines_class_info4"

FILTERED_CLASS_TABLE = "lines_class_info4_n50"

LARGE_DATA_FILE = "../../data/n50_examples_large_v01.csv"

LARGE_DATA_FILE_SORTED = "../../data/n50_examples_large_v01_sorted.csv"


## see teeb andmefaili n50 jaoks: n30 + n70

### andmetabelid

In [3]:
conn = sqlite3.connect(DATABASE)
cursor = conn.cursor()

### graafiku punktide info

In [4]:
query = f"""SELECT verb, verb_compound, morph_case, log2_ratio, level,unique_lemmas, ann_unique_lemmas, 
            not_ann_unique_lemmas, olulisus, my_tag, other_tags, annotated, not_annotated, verb_case_count
            FROM {LINE_DATA_TABLE}
            """

class_info = pd.read_sql(query, conn)
class_info

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
0,aasima,,ad,-9.965784,-,1,NaN,5.0,-,0,1,1,7,8
1,abistama,,all,-9.965784,-,1,NaN,12.0,-,0,1,1,15,16
2,aeglustama,,all,-9.965784,-,1,NaN,7.0,-,0,1,1,8,9
3,aerutama,,in,-9.965784,-,1,NaN,5.0,-,0,3,3,10,13
4,aevastama,,in,9.965784,-,1,1.0,6.0,-,1,0,1,6,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20935,õnnestuma,,ad,-1.971847,-,1313,331.0,2522.0,-,1022,4009,5031,17243,22274
20936,õppima,,in,5.233872,n90,530,463.0,985.0,0.0,5720,152,5872,5891,11763
20937,ütlema,,ad,-5.370614,n10,415,104.0,1162.0,0.0,295,12205,12500,9966,22466
20938,ütlema,,all,0.271387,n70,1131,189.0,2337.0,1.0,9354,7750,17104,40742,57846


In [13]:
class_info[(class_info["verb"]=="saama") & (class_info["verb_compound"]=="") & (class_info["morph_case"]=="in")]

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
20834,saama,,in,0.575469,n70,2161,1580.0,6384.0,1.0,13556,9097,22653,36040,58693


In [36]:
class_info[(class_info["verb"]=="suunama") & (class_info["verb_compound"]=="") & (class_info["morph_case"]=="el")]

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
18266,suunama,,el,0.192645,n70,27,14.0,96.0,0.98574,16,14,30,134,164


### võtta ainult n20 tsooni lõksud

In [23]:
filtered_class = class_info[(class_info["level"]=="n30") | (class_info["level"]=="n70")]
filtered_class = filtered_class.sort_values(["olulisus"])
filtered_class['olulisus'] = filtered_class['olulisus'].astype(float)

In [24]:
filtered_class

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
20067,nägema,ette,in,0.688056,n70,79,57.0,273.0,0.95005,87,54,141,687,828
19161,vaatama,,ill,0.597902,n70,53,39.0,201.0,0.95051,56,37,93,1075,1168
19249,nägema,välja,in,0.691878,n70,76,47.0,343.0,0.95076,63,39,102,656,758
14992,loendama,,ad,0.415037,n70,28,16.0,31.0,0.95090,24,18,42,54,96
13650,aeguma,,in,0.180572,n70,20,11.0,35.0,0.95204,17,15,32,58,90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20316,kinnitama,,el,-0.108422,n30,257,121.0,495.0,1.00000,205,221,426,1272,1698
20273,jääma,,all,-0.910959,n30,1892,651.0,4632.0,1.00000,2715,5105,7820,33204,41024
20271,saatma,,el,0.363166,n70,350,181.0,697.0,1.00000,346,269,615,1336,1951
20467,kehtima,,ad,-0.411835,n30,256,102.0,647.0,1.00000,451,600,1051,4126,5177


In [26]:
filtered_class.to_sql(FILTERED_CLASS_TABLE, conn, if_exists="replace", index=False)

615

### võtta spatial_obl tabelist näitelaused koos vajaliku infoga

In [27]:
# spatial obl_tabelist lõksud, mis on lines_class_info4_n50 tabelis
# iga lõksu kohta max 500 lemmat ja iga unikaalse lemma kohta 1 näide


query = f"""
WITH cleaned AS (
    -- Step 1 & 2: match subset table + remove rows with timex_tag NOT NULL
    SELECT
        d.head_id,
        d.row_loc as head_loc,
        d.form,
        d.lemma,
        d.verb,
        d.verb_compound,
        d.morph_case,
        d.sentence,
        d.sentence_id,
        d.timex_tag,
        d.ekilex_tag,
        d.ner_tag
    FROM spatial_obl AS d
    JOIN {FILTERED_CLASS_TABLE} AS s
      ON d.verb = s.verb
     AND d.verb_compound = s.verb_compound
     AND d.morph_case = s.morph_case
    WHERE d.timex_tag IS NULL
),

distinct_lemmas AS (
    -- Step 3 & 4: for each lemma, pick ONE sentence deterministically
    SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY verb, verb_compound, morph_case, lemma
            ORDER BY sentence_id   -- choose best or earliest sentence
        ) AS rn_per_lemma
    FROM cleaned
),

limited AS (
    -- Step 5: limit to 500 unique lemmas per (verb, verb_compound, morph_case)
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY verb, verb_compound, morph_case
            ORDER BY lemma        -- choose 500 lexicographically smallest lemmas
        ) AS rn_group_limit
    FROM distinct_lemmas
    WHERE rn_per_lemma = 1      -- keep only one sentence per lemma
)

-- Step 6: final output
SELECT
    sentence_id,
    head_id,
    head_loc, 
    verb,
    verb_compound,
    morph_case,
    lemma,
    form,
    sentence,
    timex_tag,
    ekilex_tag,
    ner_tag
FROM limited
WHERE rn_group_limit <= 500
ORDER BY verb, verb_compound, morph_case, lemma;

"""


spatial_obl_ex = pd.read_sql(query, conn)


In [28]:
spatial_obl_ex

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag
0,7156824,11512185,10,abielluma,,in,Ameerika,Ameerikas,"Samas sai viisa Helimetsa ema , kelle ainus po...",None,location,LOC
1,3567431,5744803,5,abielluma,,in,Amsterdam,Amsterdamis,Nad abiellusid 1912. aastal Amsterdamis ning e...,None,location,LOC
2,4670655,7505293,6,abielluma,,in,Aram,Aramis,"Õhku jääb küsimus , kas Aramis on kakskümmend ...",None,None,LOC
3,13486757,21571083,3,abielluma,,in,Austraalia,Austraalias,Paar abiellus Austraalias 15. novembril 1996 k...,None,location,LOC
4,14995659,23525347,3,abielluma,,in,Bagdad,Bagdadis,Mullu augustis Bagdadis abiellunud USA armee s...,None,location,LOC
...,...,...,...,...,...,...,...,...,...,...,...,...
179834,8002899,12836303,25,üürima,,abl,vanamees,vanamehelt,"Mu õde läks sinna elama , kui oli Räpina tehni...",None,alive,None
179835,12777314,20438863,3,üürima,,abl,viimane,viimaselt,Knapp üüris viimaselt oma firma tarbeks kaks p...,None,None,None
179836,3498805,5634032,11,üürima,,abl,äriomanik,äriomanikelt,Hoones leidis pikka aega peavarju Ugala teater...,None,None,None
179837,10579104,16957624,7,üürima,,abl,ühistu,ühistult,Kaks aastat üüris keskkool Hiiumaa tarbijate ü...,None,None,None


In [29]:
spatial_obl_ex.to_csv(LARGE_DATA_FILE_SORTED, encoding="utf-8", index = False,sep=",", quoting=csv.QUOTE_MINIMAL)

In [30]:
# shuffle
df = spatial_obl_ex.sample(frac=1)

In [31]:
df.to_csv(LARGE_DATA_FILE, encoding="utf-8", index = False,sep=",", quoting=csv.QUOTE_MINIMAL)

In [33]:
conn.close()

In [34]:
df2 = pd.read_csv(LARGE_DATA_FILE, encoding="utf-8", sep=",")

In [35]:
counts2 = df2.groupby(['verb','verb_compound', 'morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)
counts2

,verb,verb_compound,morph_case,count
613,ütlema,NaN,el,500
612,ütlema,NaN,all,500
577,viima,NaN,all,500
563,valitsema,NaN,ad,500
555,vaatama,NaN,el,500
...,...,...,...,...
157,keelustama,NaN,el,30
438,seisma,ees,all,29
446,soosima,NaN,all,28
484,tagurdama,otsa,ad,17
